<a href="https://colab.research.google.com/github/Liamlenguyen/LeNguyenNamHieu/blob/main/Copy_of_Ph%C3%A2n_lo%E1%BA%A1i_%E1%BA%A3nh_l%C3%A1_c%C3%A0_chua_v%E1%BB%9Bi_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phân loại ảnh lá cà chua với CNN

**Mô tả:**
Trong tác vụ này, bạn cần xây dựng một kiến trúc CNN để phân loại ảnh lá cà chua thành các loại bệnh. Dataset gồm ảnh lá cây và nhãn bệnh tương ứng, chia thành 2 tập train và test.

In [ ]:
import os
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models

In [ ]:
# you need to download and prepare the dataset
url = "https://drive.google.com/drive/folders/13VYNsm5o9NLDVOPslDPl8CzJhjRS35Uf?usp=drive_link"

In [ ]:
class TrainImageDataset(Dataset):
    def __init__(self, train, csv_file="/content/train_labels.csv", transforms=None): # add some params here
        # add some initializations here
        self.train = train

        self.transform = transform

        if csv_file:
            self.data_info = pd.read_csv("/content/train_labels.csv")
        else:
            # Nếu không có CSV, ta tự quét thư mục (giả định cấu trúc thư mục là nhãn)
            self.image_files = [f for f in os.listdir(train) if f.endswith(('.jpg', '.png', '.jpeg'))]
    def __len__(self):
        return len(self.data_info) if hasattr(self, 'data_info') else len(self.image_files) # add some logic here

    def __getitem__(self, idx):
        # change logics to load images and corresponding labels
        if hasattr(self, 'data_info'):
            img_name = os.path.join(self.train, self.data_info.iloc[idx, 0])
            label = self.data_info.iloc[idx, 1]
        else:
            img_name = os.path.join(self.train, self.image_files[idx])
            # Giả định nhãn được lấy từ tên file hoặc xử lý logic riêng
            label = 0
        image = Image.open(img_name).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    # you can change or add more transforms here
])

In [ ]:
BATCH_SIZE = 8 # you can change the batch size
train = "/content/train.zip"
labels_csv = "/content/train_labels.csv"
train_dataset = TrainImageDataset(
    train=train,
    csv_file=labels_csv,
    transforms=transform,
)  # add some args here
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
import zipfile

# Unzip train.zip
with zipfile.ZipFile('/content/train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/train.zip')

# Unzip test.zip
with zipfile.ZipFile('/content/test.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/test.zip')

print("Datasets unzipped successfully!")

BadZipFile: File is not a zip file

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install gdown

In [ ]:
# you need to download and prepare the dataset
url = "https://drive.google.com/drive/folders/13VYNsm5o9NLDVOPslDPl8CzJhjRS35Uf?usp=drive_link"
folder_id = url.split('/')[-1].split('?')[0] # Extract folder ID

import gdown
import os

print(f"Downloading files from Google Drive folder ID: {folder_id}")
output_dir = '/content/'
os.makedirs(output_dir, exist_ok=True)
gdown.download_folder(id=folder_id, output=output_dir, quiet=False, use_cookies=False)
print("Download complete. Checking downloaded files:")
print(os.listdir(output_dir))


In [ ]:
import zipfile
import os

# Create directories for unzipped content
os.makedirs('/content/train', exist_ok=True)
os.makedirs('/content/test', exist_ok=True)

# Unzip train.zip
with zipfile.ZipFile('/content/train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/train')

# Unzip test.zip
with zipfile.ZipFile('/content/test.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/test')

print("Datasets unzipped successfully!")


In [ ]:
BATCH_SIZE = 8 # you can change the batch size
train_dataset = TrainImageDataset(train="/content/train", transforms=transform)  # add some args here
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# you are free to split training dataset into train/valid and create validation dataloader here

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 5

In [ ]:
# !!! do not change anything of this cell !!!
from torch import Tensor
from typing import Type

class BasicBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        expansion: int = 1,
        downsample: nn.Module = None,
    ) -> None:
        super(BasicBlock, self).__init__()

        self.expansion = expansion
        self.downsample = downsample
        self.conv1_layer = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        self.batch_norm1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_layer = nn.Conv2d(
            out_channels,
            out_channels * self.expansion,
            kernel_size=3,
            padding=1,
            bias=False,
        )

        self.batch_norm2 = nn.BatchNorm2d(out_channels * self.expansion)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1_layer(x)
        out = self.batch_norm1(out)
        out = self.relu(out)

        out = self.conv2_layer(out)
        out = self.batch_norm2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class CNN(nn.Module):
    def __init__(
        self,
        block: Type[BasicBlock],
        img_channels: int = 3,
        num_classes: int = 10,
    ) -> None:
        super(CNN, self).__init__()
        layers = [2, 2, 2, 2]
        self.expansion = 1

        self.in_channels = 64

        self.conv_layer = nn.Conv2d(
            in_channels=img_channels,
            out_channels=self.in_channels,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        self.batch_norm = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool_layer = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer_1 = self._make_layer(block, 64, layers[0])
        self.layer_2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer_3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer_4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool_layer = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_layer = nn.Linear(512 * self.expansion, num_classes)

    def _make_layer(
        self, block: Type[BasicBlock], out_channels: int, blocks: int, stride: int = 1
    ) -> nn.Sequential:
        downsample = None
        if stride != 1:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels,
                    out_channels * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels * self.expansion),
            )
        layers = []
        layers.append(
            block(self.in_channels, out_channels, stride, self.expansion, downsample)
        )
        self.in_channels = out_channels * self.expansion

        for i in range(1, blocks):
            layers.append(
                block(self.in_channels, out_channels, expansion=self.expansion)
            )
        return nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        x = self.conv_layer(x)
        x = self.batch_norm(x)
        x = self.relu(x)
        x = self.maxpool_layer(x)

        x = self.layer_1(x)
        x = self.layer_2(x)
        x = self.layer_3(x)
        x = self.layer_4(x)

        x = self.avgpool_layer(x)
        x = torch.flatten(x, 1)
        x = self.fc_layer(x)

        return x

model = CNN(block=BasicBlock, num_classes=NUM_CLASSES).to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3) # you can change the learning rate here
criterion = nn.CrossEntropyLoss() # and loss function too

In [ ]:
from timeit import default_timer as timer

model_results = {"train_loss": [], "train_acc": []}
EPOCHS = 20 # you can change the epoch here
start_timer = timer()

for epoch in tqdm(range(EPOCHS)):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)
        # add logics to zero gradients, calculate y_pred, loss value and update optimizer
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)
        train_loss += loss.item()

    train_loss /= len(train_dataloader)
    train_acc /= len(train_dataloader)
    model_results["train_loss"].append(train_loss)
    model_results["train_acc"].append(train_acc)

    torch.save(model.state_dict(), f"model_epoch_{epoch}.pth")

    print(f"Epoch: {epoch+1} | Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")

end_timer = timer()
print(f"Total training time: {end_timer - start_timer:.3f} seconds")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, ...): # add some params here
        # add some initializations here

    def __len__(self):
        return ... # add some logic here

    def __getitem__(self, idx):
        # change logics to load images and corresponding labels
        image = None
        image_name = None

        return image, image_name

In [ ]:
BATCH_SIZE = 1 # you can change the batch size here
test_dataset = TestImageDataset(...) # add some args here
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# you can change any logics here as you prefer
all_preds = []
img_names = []

model.load_state_dict(torch.load(f"model_epoch_{EPOCHS - 1}.pth"))
model.eval()

with torch.no_grad():
    for images, image_names in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        img_names.extend(image_names)

In [ ]:
# save all_preds to CSV file here